### Setup

In [1]:
question_type = "open-ended" # available options: "open-ended"
dataset = "ptpt" # available options: "ptbr", "ptpt"
prompt_language = dataset

model_judged = "llama-3.3-70b-instruct" 

### Load Question-Answer Pairs

In [2]:
import json

with open(f"prompts/{dataset}-{question_type}-prompts-prompt-language-{prompt_language}.json", "r", encoding="utf-8") as f:
    prompts = json.load(f)

print(f"Loaded {len(prompts)} prompts from {dataset} dataset ({question_type})")

Loaded 544 prompts from ptpt dataset (open-ended)


### Load Model Responses

In [3]:
# Initialize a dict instead of a list
responses = []

responses_file = f'{model_judged}-{dataset}-open-ended-responses-prompt-language-{prompt_language}.json'

with open(f'results/{question_type}/{responses_file}', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():  # skip empty lines
            responses.append(json.loads(line))

### Add prompt instructions and format options

In [4]:
def build_prompt(model_answer: str, golden_answer: str) -> str:
    if model_answer.strip() == "":
        model_answer = "I could not provide an answer. Consider it as incorrect."
    prompt = f"""
You are an impartial judge. Your task is to evaluate the correctness of a model's 
answer to an open-ended math question by comparing it ONLY to the gold solution.

The model’s answer may use different reasoning steps or formatting than the 
gold solution; this is acceptable **as long as the final mathematical result 
is equivalent**. Focus strictly on mathematical correctness.

Instructions:
1. Compare the **final result** of the model answer to the gold answer.
2. Minor algebraic, arithmetic, or formatting differences are allowed.
3. Completely ignore writing style, explanation detail, or formatting.
4. If the final results match or are mathematically equivalent → score 1.
5. If the final results do not match → score 0.
6. Output ONLY a JSON object in this exact format:

{{
  "score": 0 or 1,
  "explanation": "short explanation of the comparison"
}}

[BEGIN DATA]
************
[Golden Answer]: {golden_answer}
************
[Model Answer]: {model_answer} 
************
[END DATA]

Provide your judgment now.
"""
    return prompt

### Loop that creates a prompt for every item in the input .json file

In [5]:
prompts_by_id = {item["id"]: item for item in prompts}

judge_prompts = []

error_flag = False

for item in responses:
    try:
        model_answer = item["raw_response"]["choice.message.content"]
    except KeyError:
        print(f"Question didn't reach model: {item.get('id')}")
        error_flag = True
    model_answer_id = item.get("id")
    golden_answer_item = prompts_by_id.get(model_answer_id)
    golden_answer = golden_answer_item["correct_answer"]
    prompt = build_prompt(model_answer, golden_answer)
    
    judge_prompts.append({
        "id": model_answer_id,
        "level": golden_answer_item.get("level"),
        "contains_latex_figure_in_question": golden_answer_item.get("contains_latex_figure_in_question"),
        "prompt": prompt,
    })

if error_flag:
    raise RuntimeError("Some questions did not reach the model. Please check the logs above.")

### Print prompts

In [6]:
for entry in judge_prompts[:4]:
    # Pretty-print with id context
    pid = entry.get("id", "unknown")
    print(f"id={pid}\n{entry.get('prompt','')}\n")

id=13

You are an impartial judge. Your task is to evaluate the correctness of a model's 
answer to an open-ended math question by comparing it ONLY to the gold solution.

The model’s answer may use different reasoning steps or formatting than the 
gold solution; this is acceptable **as long as the final mathematical result 
is equivalent**. Focus strictly on mathematical correctness.

Instructions:
1. Compare the **final result** of the model answer to the gold answer.
2. Minor algebraic, arithmetic, or formatting differences are allowed.
3. Completely ignore writing style, explanation detail, or formatting.
4. If the final results match or are mathematically equivalent → score 1.
5. If the final results do not match → score 0.
6. Output ONLY a JSON object in this exact format:

{
  "score": 0 or 1,
  "explanation": "short explanation of the comparison"
}

[BEGIN DATA]
************
[Golden Answer]: Como $$\frac{3^{31}+2^{31}}{3^{29}+2^{29}} = \frac{3^2 \left(\frac{3}{2}\right)^{29}+2^

### Store created prompts in a .json file

In [7]:
with open(f"judge-prompts/{model_judged}-{dataset}-judge-prompts-prompt-language-{prompt_language}.json", "w", encoding="utf-8") as f:
    json.dump(judge_prompts, f, ensure_ascii=False, indent=2)